# Randomized stress tests (Figure 4a/b)

## Rationale (what this experiment is proving)

This experiment is designed to market the idea that identifier mapping has **two orthogonal axes** that must be made explicit:

- **Namespace axis**: converting between Ensembl and third-party databases changes ambiguity/coverage.
- **Time axis**: converting across Ensembl releases is *time travel* through curation history.

Rather than hiding ambiguity, IDTrack treats outcome types as first-class reportable results:

- **1→0**: no valid target under the snapshot-bounded interpretation.
- **1→1**: unambiguous mapping.
- **1→n**: legitimate ambiguity (splits/merges/history), which should be reported rather than coerced.

This notebook produces the manuscript panels:

- `idtrack-manuscript/figures/fig_random_data_databases.pdf` (Fig 4a)
- `idtrack-manuscript/figures/fig_random_data_releases.pdf` (Fig 4b)

The experiment is **capability-focused** (not an accuracy benchmark):
- sample identifiers from multiple namespaces
- map them into a fixed target release/database
- summarize outcomes as **1→0 / 1→1 / 1→n**
- repeat across seeds to obtain error bars

Caching:
- Results are cached under `idtrack/docs/_notebooks/idtrack_cache/experiments/random_data/`.
- If the cache is missing, the notebook computes it (graph loading can be memory-intensive).

## Interpretation guide

- Fig 4a: outcome profiles differ by namespace; this motivates explicit ambiguity reporting.
- Fig 4b: stability of 1→1 fraction under snapshot-bounded interpretation across target releases.


In [ ]:
from __future__ import annotations

import json
import os
import random
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:  # noqa: S110
    sns = None

import sys

# Add experiments/src to sys.path
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not ((REPO_ROOT / 'idtrack').is_dir() and (REPO_ROOT / 'idtrack-manuscript').is_dir()):
    REPO_ROOT = REPO_ROOT.parent

EXPERIMENTS_SRC = REPO_ROOT / 'idtrack' / 'reproducibility' / 'experiments' / 'src'
sys.path.append(str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    MANUSCRIPT_COLORS,
    read_json,
    write_json,
    apply_rcparams,
    experiments_cache_dir,
    idtrack_cache_dir,
    load_rcparams,
    manuscript_figures_dir,
)

try:
    apply_rcparams(load_rcparams())
except Exception as e:  # noqa: S110
    print('Warning: could not apply shared rcParams:', e)

if sns is not None:
    sns.set_theme(style='whitegrid', context='paper')

plt.rcParams.update({'savefig.dpi': 300, 'figure.dpi': 140})

IDTRACK_LOCAL_REPO = idtrack_cache_dir(REPO_ROOT)
RANDOM_CACHE = experiments_cache_dir(REPO_ROOT, experiment='random_data')
MANUSCRIPT_FIGURES = manuscript_figures_dir(REPO_ROOT)

print('Repo root:', REPO_ROOT)
print('IDTRACK_LOCAL_REPO:', IDTRACK_LOCAL_REPO)
print('RANDOM_CACHE:', RANDOM_CACHE)


In [ ]:
# -------------------- Configuration --------------------

# Graph snapshot and target settings
ORGANISM_ALIAS = 'human'
SNAPSHOT_RELEASE = 114  # must be <= the cached graph snapshot you have
TARGET_RELEASE = 107
FINAL_DATABASE = None    # stay on Ensembl gene backbone

# Sampling / repeats
N_PER_NAMESPACE = 500
N_REPEATS = 6
RANDOM_SEED = 0

# Namespaces to include.
# External databases are taken from the graph and filtered through this allowlist.
EXTERNAL_DB_ALLOWLIST = {
    'HGNC Symbol',
    'UniProtKB/Swiss-Prot',
    'EntrezGene ID',
}

# Ensembl node types to include (by prefix heuristic)
ENSEMBL_PREFIXES = {
    'Ensembl gene': 'ENSG',
    'Ensembl transcript': 'ENST',
}

RESULTS_JSON = RANDOM_CACHE / (
    f"random_data_results_snapshot{SNAPSHOT_RELEASE}_target{TARGET_RELEASE}_n{N_PER_NAMESPACE}_"
    f"reps{N_REPEATS}_seed{RANDOM_SEED}.json"
)

print('RESULTS_JSON:', RESULTS_JSON)


In [ ]:
# -------------------- Compute (cache-first) --------------------

# This step loads the graph, samples IDs, runs conversions, and writes RESULTS_JSON.
# It only runs if the cache file does not already exist.

if RESULTS_JSON.exists():
    print('Using cached results:', RESULTS_JSON)
else:
    import idtrack
    from idtrack import DB

    rng = np.random.default_rng(RANDOM_SEED)

    api = idtrack.API(local_repository=str(IDTRACK_LOCAL_REPO))
    api.configure_logger()

    organism, latest = api.resolve_organism(ORGANISM_ALIAS)
    if SNAPSHOT_RELEASE > latest:
        raise ValueError(f"SNAPSHOT_RELEASE={SNAPSHOT_RELEASE} exceeds latest={latest} for {ORGANISM_ALIAS}")

    t0_all = time.perf_counter()
    api.build_graph(organism_name=organism, snapshot_release=SNAPSHOT_RELEASE, calculate_caches=True)
    g = api.track.graph

    # Precompute candidate pools once (expensive on large graphs)
    ensembl_candidates: dict[str, list[str]] = {}
    for label, prefix in ENSEMBL_PREFIXES.items():
        ensembl_candidates[label] = [n for n in g.nodes if isinstance(n, str) and n.startswith(prefix)]

    gene_candidates = ensembl_candidates.get('Ensembl gene', [n for n in g.nodes if isinstance(n, str) and n.startswith('ENSG')])
    if not gene_candidates:
        raise RuntimeError('No ENSG nodes found; cannot run release panel.')

    # Group external nodes by their database label using combined_edges.
    ext_by_db: dict[str, list[str]] = {}
    for node, att in g.nodes(data=True):
        if att.get(DB.node_type_str) != DB.nts_external:
            continue
        ce = g.combined_edges.get(node)
        if not ce:
            continue
        for db_name in ce.keys():
            ext_by_db.setdefault(db_name, []).append(node)

    external_dbs = sorted(set(ext_by_db) & set(EXTERNAL_DB_ALLOWLIST))
    print('External DBs used:', external_dbs)

    def pick_release_for_external(node: str, db_name: str) -> int | None:
        ce = g.combined_edges.get(node, {})
        sub = ce.get(db_name, {})
        releases = set()
        for relset in sub.values():
            releases |= set(relset)
        if not releases:
            return None
        return int(rng.choice(sorted(releases)))

    def pick_release_for_ensembl(node: str) -> int | None:
        try:
            ranges = g.get_active_ranges_of_id[node]
        except Exception:
            return None
        if not ranges:
            return None
        lo, hi = ranges[int(rng.integers(0, len(ranges)))]
        hi_int = int(max(g.graph.get('confident_for_release', [SNAPSHOT_RELEASE]))) if hi == float('inf') else int(hi)
        if lo > hi_int:
            return None
        return int(rng.integers(int(lo), hi_int + 1))

    def summarize(matchings: list[dict]) -> dict[str, float]:
        bins = api.classify_multiple_conversion(matchings)
        n = len(bins['input_identifiers'])

        one0 = len(bins['matching_1_to_0'])
        one1 = len(bins['matching_1_to_1']) + len(bins['alternative_target_1_to_1'])
        onen = len(bins['matching_1_to_n']) + len(bins['alternative_target_1_to_n'])

        changed = len(bins['changed_only_1_to_1']) + len(bins['changed_only_1_to_n'])

        return {
            'n': n,
            'frac_1_to_0': one0 / n if n else float('nan'),
            'frac_1_to_1': one1 / n if n else float('nan'),
            'frac_1_to_n': onen / n if n else float('nan'),
            'frac_changed': changed / n if n else float('nan'),
            'frac_changed_only_1_to_1': (len(bins['changed_only_1_to_1']) / n) if n else float('nan'),
            'frac_changed_only_1_to_n': (len(bins['changed_only_1_to_n']) / n) if n else float('nan'),
        }

    # -------------------- Fig 4a: outcome profile by namespace --------------------
    db_rows = []

    for rep in range(N_REPEATS):
        print(f"Repeat {rep+1}/{N_REPEATS}")

        # 1) Ensembl namespaces (prefix-based sampling)
        for label, candidates in ensembl_candidates.items():
            if not candidates:
                continue
            sample = rng.choice(candidates, size=min(N_PER_NAMESPACE, len(candidates)), replace=False).tolist()

            known = []
            unknown = []
            for q in sample:
                fr = pick_release_for_ensembl(q)
                if fr is None:
                    continue
                known.append(
                    api.convert_identifier(
                        q,
                        from_release=fr,
                        to_release=TARGET_RELEASE,
                        final_database=FINAL_DATABASE,
                        strategy='all',
                    )
                )
                unknown.append(
                    api.convert_identifier(
                        q,
                        from_release=None,
                        to_release=TARGET_RELEASE,
                        final_database=FINAL_DATABASE,
                        strategy='all',
                    )
                )

            db_rows.append({'namespace': label, 'mode': 'known', 'repeat': rep, **summarize(known)})
            db_rows.append({'namespace': label, 'mode': 'unknown', 'repeat': rep, **summarize(unknown)})

        # 2) External namespaces (sample external nodes grouped by DB)
        for db_name in external_dbs:
            candidates = ext_by_db.get(db_name, [])
            if not candidates:
                continue
            sample = rng.choice(candidates, size=min(N_PER_NAMESPACE, len(candidates)), replace=False).tolist()

            known = []
            unknown = []
            for q in sample:
                fr = pick_release_for_external(q, db_name)
                if fr is None:
                    continue
                known.append(
                    api.convert_identifier(
                        q,
                        from_release=fr,
                        to_release=TARGET_RELEASE,
                        final_database=FINAL_DATABASE,
                        strategy='all',
                    )
                )
                unknown.append(
                    api.convert_identifier(
                        q,
                        from_release=None,
                        to_release=TARGET_RELEASE,
                        final_database=FINAL_DATABASE,
                        strategy='all',
                    )
                )

            db_rows.append({'namespace': db_name, 'mode': 'known', 'repeat': rep, **summarize(known)})
            db_rows.append({'namespace': db_name, 'mode': 'unknown', 'repeat': rep, **summarize(unknown)})

    db_df = pd.DataFrame(db_rows)

    # -------------------- Fig 4b: 1→1 frequency vs target release --------------------
    rel_rows = []

    base_sample = rng.choice(gene_candidates, size=min(N_PER_NAMESPACE, len(gene_candidates)), replace=False).tolist()
    target_releases = list(range(min(TARGET_RELEASE, SNAPSHOT_RELEASE), SNAPSHOT_RELEASE + 1, 2))

    for rep in range(N_REPEATS):
        for to_rel in target_releases:
            matchings = []
            for q in base_sample:
                fr = pick_release_for_ensembl(q)
                if fr is None:
                    continue
                matchings.append(
                    api.convert_identifier(
                        q,
                        from_release=fr,
                        to_release=to_rel,
                        final_database=FINAL_DATABASE,
                        strategy='all',
                    )
                )
            stats = summarize(matchings)
            rel_rows.append({'to_release': to_rel, 'repeat': rep, **stats})

    rel_df = pd.DataFrame(rel_rows)

    payload = {
        'params': {
            'snapshot_release': SNAPSHOT_RELEASE,
            'target_release': TARGET_RELEASE,
            'final_database': FINAL_DATABASE,
            'n_per_namespace': N_PER_NAMESPACE,
            'n_repeats': N_REPEATS,
            'random_seed': RANDOM_SEED,
            'external_db_allowlist': sorted(EXTERNAL_DB_ALLOWLIST),
            'ensembl_prefixes': ENSEMBL_PREFIXES,
        },
        'timing_seconds_total': time.perf_counter() - t0_all,
        'db_df': db_df.to_dict(orient='records'),
        'rel_df': rel_df.to_dict(orient='records'),
    }

    write_json(payload, RESULTS_JSON)
    print('Wrote:', RESULTS_JSON)


In [ ]:
# -------------------- Load cached results --------------------

if not RESULTS_JSON.exists():
    raise FileNotFoundError(
        f"No cached results found at {RESULTS_JSON}. Run the notebook from the top to generate the cache."
    )

payload = read_json(RESULTS_JSON)
print('Loaded:', RESULTS_JSON)

db_df = pd.DataFrame(payload['db_df'])
rel_df = pd.DataFrame(payload['rel_df'])

display(db_df.head())
display(rel_df.head())


In [ ]:
# -------------------- Plot Fig 4a (databases / namespaces) --------------------

if payload is None:
    print('No payload loaded; skipping plot.')
else:
    db_df = pd.DataFrame(payload['db_df'])

    # Aggregate across repeats
    agg = (
        db_df.groupby(['namespace', 'mode'])[['frac_1_to_0', 'frac_1_to_1', 'frac_1_to_n']]
        .agg(['mean', 'std'])
        .reset_index()
    )

    # Keep a stable order
    namespaces = list(dict.fromkeys(db_df['namespace'].tolist()))

    # Build stacked bars: two bars per namespace (known vs unknown)
    fig, ax = plt.subplots(1, 1, figsize=(7, 5))

    x = np.arange(len(namespaces))
    width = 0.35

    def _bar(mode: str, offset: float):
        sub = agg[agg['mode'] == mode].set_index('namespace')
        m0 = sub[('frac_1_to_0', 'mean')].reindex(namespaces).fillna(0).values
        m1 = sub[('frac_1_to_1', 'mean')].reindex(namespaces).fillna(0).values
        mn = sub[('frac_1_to_n', 'mean')].reindex(namespaces).fillna(0).values

        ax.bar(x + offset, m1, width, label=f'1→1 ({mode})', color=MANUSCRIPT_COLORS['1→1'])
        ax.bar(x + offset, mn, width, bottom=m1, label=f'1→n ({mode})', color=MANUSCRIPT_COLORS['1→n'])
        ax.bar(x + offset, m0, width, bottom=m1 + mn, label=f'1→0 ({mode})', color=MANUSCRIPT_COLORS['1→0'])

    _bar('known', -width/2)
    _bar('unknown', +width/2)

    ax.set_xticks(x)
    ax.set_xticklabels(namespaces, rotation=45, ha='right')
    ax.set_ylabel('Fraction of queries')
    ax.set_ylim(0, 1)
    ax.set_title('Randomized stress test across namespaces')

    # De-duplicate legend entries
    handles, labels = ax.get_legend_handles_labels()
    uniq = dict(zip(labels, handles))
    ax.legend(uniq.values(), uniq.keys(), loc='upper right', frameon=True)

    fig.tight_layout()

    out_fig = MANUSCRIPT_FIGURES / 'fig_random_data_databases.pdf'
    fig.savefig(out_fig, bbox_inches='tight')
    print('Saved:', out_fig)


In [ ]:
# -------------------- Plot Fig 4b (target release axis) --------------------

if payload is None:
    print('No payload loaded; skipping plot.')
else:
    rel_df = pd.DataFrame(payload['rel_df'])

    agg = rel_df.groupby('to_release')[['frac_1_to_1']].agg(['mean', 'std']).reset_index()

    fig, ax = plt.subplots(1, 1, figsize=(6, 4))
    ax.errorbar(
        agg['to_release'],
        agg[('frac_1_to_1', 'mean')],
        yerr=agg[('frac_1_to_1', 'std')],
        fmt='-o',
        color=MANUSCRIPT_COLORS['1→1'],
        ecolor=MANUSCRIPT_COLORS['1→1'],
        capsize=3,
        lw=1.5,
        ms=4,
    )
    ax.set_xlabel('Target Ensembl release')
    ax.set_ylabel('1→1 fraction')
    ax.set_ylim(0, 1)
    ax.set_title('Stability under snapshot-bounded interpretation')

    fig.tight_layout()

    out_fig = MANUSCRIPT_FIGURES / 'fig_random_data_releases.pdf'
    fig.savefig(out_fig, bbox_inches='tight')
    print('Saved:', out_fig)


In [ ]:
# -------------------- Optional: drift diagnostics (extra) --------------------

# This figure is not referenced in the main manuscript by default, but is useful for marketing:
# it visualizes how often IDs *change* under time travel (changed-only 1→1 and 1→n combined).

if payload is None:
    print('No payload loaded; skipping drift plot.')
else:
    db_df = pd.DataFrame(payload['db_df'])

    agg = (
        db_df.groupby(['namespace', 'mode'])[['frac_changed']]
        .agg(['mean', 'std'])
        .reset_index()
    )

    namespaces = list(dict.fromkeys(db_df['namespace'].tolist()))

    fig, ax = plt.subplots(1, 1, figsize=(7, 4.5))

    x = np.arange(len(namespaces))
    width = 0.35

    def _bar(mode: str, offset: float, color: str):
        sub = agg[agg['mode'] == mode].set_index('namespace')
        m = sub[('frac_changed', 'mean')].reindex(namespaces).fillna(0).values
        s = sub[('frac_changed', 'std')].reindex(namespaces).fillna(0).values
        ax.bar(x + offset, m, width, yerr=s, capsize=3, color=color, label=mode)

    _bar('known', -width / 2, MANUSCRIPT_COLORS['1→1'])
    _bar('unknown', +width / 2, MANUSCRIPT_COLORS['1→n'])

    ax.set_xticks(x)
    ax.set_xticklabels(namespaces, rotation=45, ha='right')
    ax.set_ylabel('Fraction changed (changed-only 1→1 + 1→n)')
    ax.set_ylim(0, 1)
    ax.set_title('Identifier drift signal across namespaces')
    ax.legend(title='from_release')

    fig.tight_layout()

    out_fig = MANUSCRIPT_FIGURES / 'fig_random_data_drift.pdf'
    fig.savefig(out_fig, bbox_inches='tight')
    print('Saved:', out_fig)


In [ ]:
# -------------------- Optional: combined multi-panel figure (marketing) --------------------

# This figure is not required for the main manuscript plan, but is useful when you want to show
# the two stress-test panels together in a single export.

if payload is None:
    print('No payload loaded; skipping combined figure.')
else:
    db_df = pd.DataFrame(payload['db_df'])
    rel_df = pd.DataFrame(payload['rel_df'])

    # Aggregates for export
    agg_db = (
        db_df.groupby(['namespace', 'mode'])[['frac_1_to_0', 'frac_1_to_1', 'frac_1_to_n']]
        .agg(['mean', 'std'])
        .reset_index()
    )
    agg_rel = rel_df.groupby('to_release')[['frac_1_to_1']].agg(['mean', 'std']).reset_index()

    (RANDOM_CACHE / 'random_data_db_aggregated.csv').write_text(agg_db.to_csv(index=False))
    (RANDOM_CACHE / 'random_data_release_aggregated.csv').write_text(agg_rel.to_csv(index=False))

    # Combined plot
    fig, (axA, axB) = plt.subplots(1, 2, figsize=(12.5, 4.2), constrained_layout=True)

    # Panel A (namespace axis; two modes per namespace)
    namespaces = list(dict.fromkeys(db_df['namespace'].tolist()))
    x = np.arange(len(namespaces))
    width = 0.35

    def _stack(mode: str, offset: float):
        sub = agg_db[agg_db['mode'] == mode].set_index('namespace')
        m0 = sub[('frac_1_to_0', 'mean')].reindex(namespaces).fillna(0).values
        m1 = sub[('frac_1_to_1', 'mean')].reindex(namespaces).fillna(0).values
        mn = sub[('frac_1_to_n', 'mean')].reindex(namespaces).fillna(0).values
        axA.bar(x + offset, m1, width, color=MANUSCRIPT_COLORS['1→1'])
        axA.bar(x + offset, mn, width, bottom=m1, color=MANUSCRIPT_COLORS['1→n'])
        axA.bar(x + offset, m0, width, bottom=m1 + mn, color=MANUSCRIPT_COLORS['1→0'])

    _stack('known', -width / 2)
    _stack('unknown', +width / 2)

    axA.set_xticks(x)
    axA.set_xticklabels(namespaces, rotation=35, ha='right')
    axA.set_ylabel('Fraction of queries')
    axA.set_ylim(0, 1)
    axA.set_title('Outcome profiles across namespaces')

    # Legend (single, semantic)
    axA.legend(
        handles=[
            plt.Rectangle((0, 0), 1, 1, color=MANUSCRIPT_COLORS['1→0']),
            plt.Rectangle((0, 0), 1, 1, color=MANUSCRIPT_COLORS['1→1']),
            plt.Rectangle((0, 0), 1, 1, color=MANUSCRIPT_COLORS['1→n']),
        ],
        labels=['1→0', '1→1', '1→n'],
        loc='upper right',
        frameon=True,
        title='Outcome',
    )

    # Panel B (time axis)
    axB.errorbar(
        agg_rel['to_release'],
        agg_rel[('frac_1_to_1', 'mean')],
        yerr=agg_rel[('frac_1_to_1', 'std')],
        fmt='-o',
        color=MANUSCRIPT_COLORS['1→1'],
        ecolor=MANUSCRIPT_COLORS['1→1'],
        capsize=3,
        lw=1.5,
        ms=4,
    )
    axB.set_xlabel('Target Ensembl release')
    axB.set_ylabel('1→1 fraction')
    axB.set_ylim(0, 1)
    axB.set_title('Stability under snapshot-bounded interpretation')

    out_fig = MANUSCRIPT_FIGURES / 'fig_random_data_combined.pdf'
    fig.savefig(out_fig, bbox_inches='tight')
    print('Saved:', out_fig)
